# Flamanville 3 EPR Grid Connection, 2024-12-21

Historical reproduction (see the [bertellis-flow-diagnostics](../bertellis-flow-diagnostics/bertellis_flow_diagnostics.ipynb) notebook) only proves a platform can look backward at known moments. This notebook points the same reusable functions &ndash; `hourly_border_counts`, `before_after_convergence` &ndash; at a real, ongoing structural change: France's first new nuclear unit in 25 years (1,630 MW) connecting to the grid. ([EDF press release](https://www.edf.fr/en/the-edf-group/dedicated-sections/journalists/all-press-releases/update-on-the-flamanville-epr-the-reactor-produces-its-first-electrons-on-the-national-electricity-grid))

The window is 8 days (event date &plusmn; 4 days) at hourly resolution &ndash; wide enough to see a trend, narrow enough that the API doesn't auto-downgrade past hourly.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.

**Before you run this:** sign up at [voltpath.ai](https://voltpath.ai) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY` (or leave `.env` empty and you'll be prompted).


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from common import (
    before_after_convergence,
    before_after_convergence_chart,
    congestion_ci_chart,
    fetch_flows,
    save_chart,
)

In [ ]:
FLAMANVILLE_EVENT = {
    "name": "Flamanville 3 EPR grid connection",
    "slug": "flamanville-3",
    "date": "2024-12-21",
    "date_from": "2024-12-17",
    "date_to": "2024-12-25",
}

### Congested / counter-intuitive borders

In [ ]:
flamanville_event = FLAMANVILLE_EVENT
flamanville_df = fetch_flows(flamanville_event["date_from"], flamanville_event["date_to"], resolution="1H")
fig = congestion_ci_chart(flamanville_df, flamanville_event)
save_chart(
    fig,
    f"{flamanville_event['slug']}-congestion-ci-{flamanville_event['date_from']}-to-{flamanville_event['date_to']}",
)
fig.show()

### Did coupling reliability actually change?

In [ ]:
comparison = before_after_convergence(flamanville_df, flamanville_event["date"])
fig = before_after_convergence_chart(comparison, flamanville_event)
save_chart(
    fig,
    f"{flamanville_event['slug']}-convergence-before-after-{flamanville_event['date_from']}-to-{flamanville_event['date_to']}",
)
fig.show()

### Careful: this is probably not Flamanville

The before/after convergence numbers above move in the "expected" direction, but no primary source (RTE, EDF, ASNR, ACER, JAO, or trade press) attributes any December 2024 congestion or convergence shift to Flamanville specifically. Flamanville connected 2024-12-21 at only ~100 MW and stayed regulator-capped at &le;25% power (~408 MW) by ASNR until 2025-01-31 &ndash; it didn't reach full 1,630 MW output until 2025-12-14, a year later. RTE's own 2024 Bilan &Eacute;lectrique attributes that period's France-cheaper-than-Germany price flip and export records to fleet-wide nuclear recovery (279.0 TWh in 2022 &rarr; 361.7 TWh in 2024, availability 71.5%) plus a record hydro year &ndash; i.e. the *existing* ~56-reactor fleet, not the one new unit still ramping up. Good reminder that a mechanical before/after check can find a real-looking pattern with the wrong cause behind it.


## Takeaways

- The same functions used to reproduce a year-old published analysis (see [bertellis-flow-diagnostics](../bertellis-flow-diagnostics/bertellis_flow_diagnostics.ipynb)) turn a historical curiosity into a live monitoring tool: point them at any dated event and the before/after shift shows up mechanically.
- A mechanical before/after check can find a real-looking pattern with the wrong cause behind it: no primary source attributes December 2024's congestion/convergence shift to Flamanville, which was still capped at ~25% power a month after connecting.

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).
